# End-to-end LUT with differential-state input

Continuation of `bpam_lut.ipynb`, using its trained LUT/RX checkpoints read-only.
The new table has **2048 rows and two drive values per row**:
five unchanged amplitude bits and six consecutive differential states.

The state recursion is `c1[k] = c1[k-1] XOR b1[k]`.
The LUT sees `[b0[k-2], ..., b0[k+2], c1[k-3], ..., c1[k+2]]`.
The receiver is still trained against **raw information bits**.

Initialization reconstructs each raw phase bit by XOR of consecutive states.
It reproduces the source LUT exactly, including boundary padding. No BPAM
constellation or polarity is imposed. The two initially equivalent rows for each
raw context are subsequently free to differ.

The channel and all filter/DAC/ADC settings come from the selected source checkpoint.
The digital surrogate remains frozen. `FORCE_RETRAIN` reruns this new LUT/RX
fine-tuning stage only; upstream models are immutable prerequisites here.
To change the upstream training, use `bpam_lut.ipynb` first and select its new checkpoint.

Default: 5000 additional steps, one seed, not a convergence claim.
All new files go to `results/bpam_lut_differential/`.
The BER and eye cells use the digital surrogate, as in the source notebook.


## 1. Setup and knobs

In [ ]:
from pathlib import Path
import os
import sys
PROJECT_ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents)
                    if (p / "functions" / "config.py").is_file())
os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT / "functions"))
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"
import json
import hashlib
import numpy as np
import torch
import matplotlib.pyplot as plt
from copy import deepcopy
from scenarios import OSNR_OFFSET_DB
from train import evaluate
from differential_lut import DifferentialLookupTransmitter, load_raw_lut
from lut_training import train_lut
from simulated_surrogate import file_digest
%matplotlib inline

device = "cuda" if torch.cuda.is_available() else "cpu"
ADC_BITS = [8, 6, 5]
LUT_STEPS = 5000
LUT_SEED = 20260917
FORCE_RETRAIN = False
# Explicit source files prevent accidental selection of a different experiment.
SOURCE_CHECKPOINTS = {
    8: Path("results/bpam_lut/checkpoints/lut_adc8_8e550a0954f63234.pt"),
    6: Path("results/bpam_lut/checkpoints/lut_adc6_03d5a5b95443d051.pt"),
    5: Path("results/bpam_lut/checkpoints/lut_adc5_60d8e0a4e823c000.pt"),
}
OSNR_GRID = list(range(11, 21))
EBN0_FOR_OSNR = [o - OSNR_OFFSET_DB for o in OSNR_GRID]
def num_symbols(ebn0_db):
    return 4_000_000 if ebn0_db <= 14.0 else 16_000_000
WORK = Path("results/bpam_lut_differential")
WORK.mkdir(parents=True, exist_ok=True)
print("device:", device)


## 2. Exact conversion and joint differential-LUT/RX training

All 2048 rows are initialized from the trained 1024-row source table.
The bounded initialization check includes single-symbol and short sequences.
The extra left state is zero before the block; beyond the right edge the last
state is held, equivalent to zero padding the original raw bits.
No differential decoder is added at RX. Independent bit/noise blocks are used in training.


In [ ]:
def load_or_train(adc_bits, force_retrain=False):
    source = SOURCE_CHECKPOINTS[adc_bits]
    if not source.is_file():
        raise FileNotFoundError(f"Run bpam_lut first and select its checkpoint: {source}")
    raw, ch, rx, cfg = load_raw_lut(source, device)
    source_metadata = torch.load(source, map_location="cpu", weights_only=True)["metadata"]
    train_ebn0_db = source_metadata["train_ebn0_db"]
    tx = DifferentialLookupTransmitter.from_raw_lookup(raw)
    generator = torch.Generator(device=device).manual_seed(12345)
    with torch.no_grad():
        for length in (1, 2, 3, 5, 8192):
            bits = torch.randint(0, 2, (2, length), generator=generator, device=device)
            assert torch.equal(tx.symbol_drive_levels(bits), raw.symbol_drive_levels(bits))
            assert torch.equal(tx(bits), raw(bits))
    metadata = {
        "source_sha256": file_digest(source), "config": vars(cfg),
        "representation": "five raw amplitude bits and six differential phase states",
        "raw_rx_targets": True, "initial_phase_state": 0,
        "digital_surrogate_sha256": source_metadata["digital_surrogate_sha256"],
        "steps": LUT_STEPS, "seed": LUT_SEED, "train_ebn0_db": train_ebn0_db,
        "source_code": {name: file_digest(Path("functions") / f"{name}.py")
                        for name in ("differential_lut", "lut_training", "tx_lut")},
    }
    key = hashlib.sha256(json.dumps(metadata, sort_keys=True).encode()).hexdigest()[:16]
    path = WORK / "checkpoints" / f"differential_adc{adc_bits}_{key}.pt"
    path.parent.mkdir(parents=True, exist_ok=True)
    if path.exists() and not force_retrain:
        state = torch.load(path, map_location=device, weights_only=True)
        if state["metadata"] != metadata:
            raise RuntimeError("Incompatible differential LUT checkpoint")
        tx.load_state_dict(state["tx"])
        rx.load_state_dict(state["rx"])
        print(f"Loaded {path}")
    else:
        history = train_lut(tx, ch, rx, cfg, LUT_STEPS, train_ebn0_db, LUT_SEED)
        temporary = path.with_suffix(".tmp")
        torch.save({"tx": tx.state_dict(), "rx": rx.state_dict(),
                    "metadata": metadata, "history": history}, temporary)
        temporary.replace(path)
        print(f"Saved {path}")
    values = tx.embedding.weight.detach().cpu().numpy()
    table_path = WORK / "tables" / f"differential_adc{adc_bits}_{key}.txt"
    table_path.parent.mkdir(parents=True, exist_ok=True)
    np.savetxt(table_path, values, fmt="%.9g",
               header="Two voltages before filters/DAC per row. MSB-first address:\n"
                      "[b0[k-2]..b0[k+2], c1[k-3]..c1[k+2]]; c1 is cumulative XOR of b1.")
    assert all(not p.requires_grad and p.grad is None for p in ch.parameters())
    return tx.eval(), ch.eval(), rx.eval(), cfg

nets = {}
for adc in ADC_BITS:
    nets[adc] = load_or_train(adc, FORCE_RETRAIN)
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
print("Ready:", list(nets))


## 3. Measure BER versus OSNR


In [ ]:
results = {"osnr": {}}

for adc, (tx, ch, rx, cfg) in nets.items():
    results["osnr"][adc] = [evaluate(tx, ch, rx, cfg, [e], num_symbols(e), device)[0]
                            for e in EBN0_FOR_OSNR]
    torch.cuda.empty_cache()
    print(f"N_ADC={adc}  done")

print("\nOSNR [dB]  " + " ".join(f"{o:>9d}" for o in OSNR_GRID))
for adc in ADC_BITS:
    print(f"N_ADC={adc}   " + " ".join(f"{b:9.2e}" for b in results["osnr"][adc]))


## 4. Load the historical BPAM + adaptive-equalizer reference

This is the same MATLAB reference file used by `bpam_complex.ipynb`, not a new
digital surrogate-transfer control. It is read only and is not regenerated here.


In [ ]:
REF = os.path.join("results", "paper", "bpam_ae_reference_osnrgrid.txt")
baseline = {}
for line in open(REF):
    if line.startswith("#") or not line.strip():
        continue
    p = line.split()
    nb = int(p[0])
    baseline.setdefault(nb, {"ebn0": [], "osnr": [], "ber": []})
    baseline[nb]["ebn0"].append(float(p[2]))
    baseline[nb]["osnr"].append(float(p[3]))
    baseline[nb]["ber"].append(float(p[4]))
print("baseline loaded for N_ADC:", sorted(baseline))


## 5. Column files (the format shared with the co-authors)


In [ ]:
OUT = str(WORK)
os.makedirs(OUT, exist_ok=True)

HEADER = (
    "# End-to-end autoencoder trained and tested through a fitted hybrid digital surrogate, C-band 10.238 km, MZM ER 30 dB, TX 20 GHz rect, WSS optical filter,\n"
    "# RX 10 GHz Gaussian, N_DAC = 6, 2 samples/symbol. RX-DSP = fully connected 64-128-32 with\n"
    "# sigmoid output, context window 6 symbols (12 samples); TX-DSP = trainable LUT (2048 rows, 2 voltages per row).\n"
    "# OSNR = Eb/N0 + 2.05 dB (signal 1 pol, ASE 2 pol, Bref 12.5 GHz).\n"
    "\n"
    f"# Initialized from trained raw-bit LUT; {LUT_STEPS} differential-LUT/RX steps; raw RX targets.\n"
    "# columns:\n")

def save(axis, grid, axis_name, fname):
    heads = [f"{axis_name}[dB]"] + [f"E2E_adc{nb}" for nb in ADC_BITS] + \
            [f"BPAM_AE_adc{nb}" for nb in ADC_BITS]
    cols = [results[axis][nb] for nb in ADC_BITS] + [[dict(zip(baseline[nb]["osnr"], baseline[nb]["ber"]))[float(x)] for x in grid] for nb in ADC_BITS]
    with open(os.path.join(OUT, fname), "w") as f:
        f.write(HEADER)
        f.write("# " + "  ".join(f"{h:>16s}" for h in heads)[2:] + "\n\n")
        for i, x in enumerate(grid):
            f.write(f"{x:8.2f}  " + "  ".join(f"{c[i]:16.4e}" for c in cols) + "\n")
    print("written", os.path.join(OUT, fname))

save("osnr", OSNR_GRID, "OSNR", "bpam_lut_differential.txt")


## 6. Figures

Solid: end-to-end. Dashed: BPAM + adaptive equalizer, same `N_ADC` (same color).


In [ ]:
STYLE = {8: ("o", "#4393c3"), 6: ("s", "#d6604d"), 5: ("^", "#8c510a")}

def figure(axis, grid, xlabel, bkey, fname):
    fig, ax = plt.subplots(figsize=(7.6, 5.6))
    for nb in ADC_BITS:
        mk, c = STYLE[nb]
        pts = [(x, y) for x, y in zip(baseline[nb][bkey], baseline[nb]["ber"]) if y > 0]
        ax.semilogy([p[0] for p in pts], [p[1] for p in pts], "--" + mk, color=c, lw=1.6,
                    ms=5, mfc="none", label=f"BPAM + adaptive eq. $N_{{ADC}}={nb}$")
    for nb in ADC_BITS:
        mk, c = STYLE[nb]
        ax.semilogy(grid, np.maximum(results[axis][nb], 1e-7), "-" + mk, color=c, lw=2,
                    ms=5, label=f"E2E differential LUT $N_{{ADC}}={nb}$")
    ax.set_xlim(min(grid), max(grid)); ax.set_ylim(1e-5, 1e-1); ax.set_xticks(grid)
    ax.set_xlabel(xlabel, fontsize=12); ax.set_ylabel("BER", fontsize=12)
    ax.grid(True, which="both", alpha=0.3)
    ax.legend(fontsize=8.5, loc="lower left", ncol=2)
    ax.set_title("TX:20GHz rect, RX:10GHz Gauss - 10km MZM $N_{DAC}=6$", fontsize=11)
    fig.tight_layout()
    fig.savefig(os.path.join(OUT, fname + ".pdf"), bbox_inches="tight")
    fig.savefig(os.path.join(OUT, fname + ".png"), bbox_inches="tight", dpi=150)
    plt.show()

figure("osnr", OSNR_GRID, "OSNR [dB]", "osnr", "bpam_lut_differential")


## 6b. Noiseless photocurrent eye through the fitted digital surrogate

The waveform is taken after the digital surrogate photodetector response and the RX
Gaussian filter, before the ADC. No matched filter is applied. Interpolation
is for display only; it does not turn the 2-sps simulation into a high-rate simulation.


In [ ]:
import torch.nn.functional as F
from scipy.signal import resample_poly
from train import random_bits
from transmitter import bits_to_symbols
from utils import decision_phase_by_separation

def draw_eye(ax, signal, samples_per_symbol, phase, title, fine_sps=40, skip=100, num_symbols=4000):
    fine = resample_poly(signal[:(skip + num_symbols + 8) * samples_per_symbol],
                         fine_sps, samples_per_symbol)
    offset = int(round(phase * fine_sps / samples_per_symbol))
    centers = (skip + np.arange(num_symbols)) * fine_sps + offset
    keep = (centers - fine_sps >= 0) & (centers + fine_sps < len(fine))
    centers = centers[keep]
    window = np.arange(-fine_sps, fine_sps)[:, None] + centers[None, :]
    ax.plot(np.arange(-fine_sps, fine_sps) / fine_sps, fine[window],
            color=(0.85, 0.33, 0.10, 0.03), linewidth=0.4)
    ax.axvline(0, color="k", linestyle=":", linewidth=1)
    ax.set_xlim(-1, 1)
    ax.grid(True, alpha=0.3)
    ax.set_xlabel("Symbol time")
    ax.set_title(title)

fig, axes = plt.subplots(1, len(nets), figsize=(5.4 * len(nets), 4.6), sharey=True)
axes = np.atleast_1d(axes)
for ax, (adc, (tx, ch, rx, cfg)) in zip(axes, nets.items()):
    sps = cfg.samples_per_symbol_sim
    torch.manual_seed(7)
    with torch.no_grad():
        bits = random_bits(cfg.bits_per_symbol, 6000, device)
        symbols = bits_to_symbols(bits, cfg.bits_per_symbol).cpu().numpy()
        p = ch(tx(bits), ase_ebn0_db=None)                    # noiseless photocurrent (post-photodiode)
        if rx.rx_gauss is not None:                           # 10 GHz RX Gaussian (the RX electrical filter)
            p = F.conv1d(p.view(1, 1, -1), rx.rx_gauss, padding=rx.rx_gauss.shape[-1] // 2).view(-1)
        photocurrent = p.cpu().numpy()
    phase, _ = decision_phase_by_separation(photocurrent, sps, symbols, skip=100)
    title = "ideal ADC" if adc is None else f"$N_{{ADC}}={adc}$"
    draw_eye(ax, photocurrent, sps, phase, title)
axes[0].set_ylabel("photocurrent")
fig.suptitle("Noiseless eye of the photocurrent (post-photodiode + 10 GHz RX Gaussian)\n"
             "-- what the receiver samples", fontsize=11)
fig.tight_layout()
fig.savefig(os.path.join(OUT, "bpam_lut_differential_eyes.png"), dpi=150, bbox_inches="tight")
plt.show()


## 7. Horizontal gain vs the reference, at equal ADC resolution


In [ ]:
def crossing(x, y, target):
    x = np.asarray(x, float)
    ly = np.log10(np.maximum(np.asarray(y, float), 1e-12))
    lt = np.log10(target)
    for i in range(len(x) - 1):
        if (ly[i] - lt) * (ly[i + 1] - lt) <= 0:
            return x[i] + (x[i + 1] - x[i]) * (lt - ly[i]) / (ly[i + 1] - ly[i])
    return np.nan

for target in (1e-2, 1e-3):
    print(f"\nBER = {target:.0e}   (OSNR axis)")
    for nb in ADC_BITS:
        bx = [x for x, y in zip(baseline[nb]["osnr"], baseline[nb]["ber"]) if y > 0]
        by = [y for y in baseline[nb]["ber"] if y > 0]
        gb = crossing(bx, by, target)
        ge = crossing(OSNR_GRID, results["osnr"][nb], target)
        print(f"  N_ADC={nb}:  baseline @ {gb:5.2f} dB   E2E @ {ge:5.2f} dB"
              f"   ->  gain {gb - ge:+.2f} dB")
